In [0]:
CREATE OR REPLACE TABLE datawerhouse1.silver.customers_txt AS
SELECT
    trim(customer_id) AS customer_id,
    -- 1 convert 'credit_term_days' to an integer
    cast(credit_terms_days AS INT) AS credit_term_days,
    -- 2 convert 'annual_revenue_potential' to a double
    cast(annual_revenue_potential AS DOUBLE) AS annual_revenue_potential,
    -- include customer_name, customer_type, primary_freight_type without any change
    customer_name,
    customer_type,
    primary_freight_type,
    -- 'inticap' is not a valid function; use 'initcap' for capitalizing each word
    initcap(account_status) AS account_status,
    cast(contract_start_date AS DATE) AS contract_start_date
FROM datawerhouse1.bronze.customers_txt
WHERE TRIM(customer_id) IS NOT NULL AND TRIM(customer_name) !='';
SELECT COUNT(*) FROM datawerhouse1.silver.customers_txt;
DESCRIBE TABLE datawerhouse1.silver.customers_txt;
describe datawerhouse1.bronze.customers_txt;
SELECT * FROM datawerhouse1.silver.customers_txt

In [0]:
DESCRIBE TABLE datawerhouse1.bronze.delivery_events;
CREATE OR REPLACE TABLE datawerhouse1.silver.delivery_events AS
SELECT
    trim(event_id) AS event_id,
    trim(load_id) AS load_id,
    trim(trip_id) AS trip_id,
    upper(trim(event_type)) AS event_type,
    trim(facility_id) AS facility_id,
    -- ensuring timestamps are properly handled
    scheduled_datetime, 
    actual_datetime,
    -- business logic: calculate delay in minutes
    DATEDIFF(MINUTE, scheduled_datetime, actual_datetime) AS delay_minutes,
    -- standard numeric type
    CAST(detention_minutes AS INT) AS detention_minutes,
    on_time_flag,
    -- geography clean_up
    upper(trim(location_city)) AS location_city,
    upper(trim(location_state)) AS location_state
FROM datawerhouse1.bronze.delivery_events
WHERE trim(event_id) IS NOT NULL;
SELECT * FROM datawerhouse1.silver.delivery_events

In [0]:
describe table datawerhouse1.bronze.driver_monthly_metrics;
create or replace table datawerhouse1.silver.driver_monthly_metrics as
with ranked_data as (
    select
        trim(driver_id) as driver_id,
        month,
        trips_completed,
        total_miles,
        total_revenue,
        round(average_mpg,2) AS average_mpg,
        round(total_fuel_gallons,2) AS total_fuel_gallons,
        round(on_time_delivery_rate,4) as on_time_delivery_rate,
        round(average_idle_hours,2) as average_idle_hours,
        row_number() over (
            partition by trim(driver_id), month
            order by total_revenue desc, trips_completed desc
        ) as row_num
    from datawerhouse1.bronze.driver_monthly_metrics
    where trim(driver_id) is not null
)
select
    driver_id,
    month,
    trips_completed,
    total_miles,
    total_revenue,
    average_mpg,
    total_fuel_gallons,
    on_time_delivery_rate,
    average_idle_hours
from ranked_data
where row_num = 1;
select * from datawerhouse1.bronze.driver_monthly_metrics

In [0]:
describe table datawerhouse1.bronze.drivers

In [0]:
CREATE OR REPLACE TABLE datawerhouse1.silver.drivers AS
WITH ranked_data AS (
    SELECT
        trim(driver_id) AS driver_id,
        initcap(trim(first_name)) AS first_name,
        initcap(trim(last_name)) AS last_name,
        hire_date,
        -- handle malformed termination_date gracefully: if NULL or empty, set to 'Active', else 'Terminated: <date>'
        CASE 
            WHEN termination_date IS NULL OR trim(termination_date) = '' THEN 'Active'
            ELSE concat('Terminated: ', cast(termination_date as string))
        END AS termination_date,
        upper(trim(license_number)) AS license_number,
        upper(trim(license_state)) AS license_state,
        date_of_birth,
        home_terminal,
        initcap(trim(employment_status)) AS employment_status,
        cdl_class,
        years_experience,
        row_number() OVER (PARTITION BY trim(driver_id) ORDER BY driver_id ASC) AS row_num
    FROM datawerhouse1.bronze.drivers
    WHERE trim(driver_id) IS NOT NULL
)
SELECT
    driver_id,
    first_name,
    last_name,
    hire_date,
    termination_date,
    license_number,
    license_state,
    date_of_birth,
    home_terminal,
    employment_status,
    cdl_class,
    years_experience
FROM ranked_data
WHERE row_num = 1;
select * from datawerhouse1.silver.drivers

In [0]:
create or replace table datawerhouse1.silver.facilities as
select
  -----cleanid for joins
  trim(facility_id) as facility_id,
  ----EAST HUB to East Hub
  initcap(trim(facility_name)) as facility_name,
   ----standardize type-----
  upper(trim(facility_type)) as facility_type,
  ----standardize city-----
  upper(trim(city)) as city,
  ------standardize state-----------
  upper(trim(state)) as state,
  latitude,
  longitude,
  dock_doors,
  operating_hours
from datawerhouse1.bronze.facilities
where facility_id is not null;
select * from datawerhouse1.silver.facilities;
describe table datawerhouse1.bronze.facilities

In [0]:
create or replace table datawerhouse1.silver.fuel_purchases as
SELECT
----remove space
trim(fuel_purchase_id) as fuel_purchase_id,
trim(trip_id) as trip_id,
trim(truck_id) as truck_id,
trim(driver_id) as driver_id,
case 
    when trim(truck_id) is null or trim(driver_id) is null then 'Unknown'
    else 'Known'
end as Is_Known,
------alredy timestamp
purchase_date,
--- convert city to upper case from lower case----
upper(trim(location_city)) as city,
----convert state to upper case from lower case----
upper(trim(location_state)) as state,
-----clean up decimal
round(gallons,2) as gallons,
round(price_per_gallon,2) as price_per_gallon,
round(total_cost,2) as total_cost,
fuel_card_number
FROM datawerhouse1.bronze.fuel_purchases;

select * from datawerhouse1.silver.fuel_purchases
where fuel_purchase_id = 'FUEL00000002';
select * from datawerhouse1.silver.fuel_purchases


In [0]:
create or replace table datawerhouse1.silver.loads as
SELECT
----remove space
trim(load_id) as load_id,
trim(customer_id) as customer_id,
trim(route_id) as route_id,
-----already date
load_date,
--DRY_VAN to Dry Van
initcap(trim(load_type)) as load_type,
---already done
weight_lbs,
pieces,
----clean up to decimal
round(revenue,2) as revenue,
round(fuel_surcharge,2) as fuel_surcharge,
----already done
accessorial_charges,
-----convert to the upper case first letter only
initcap(trim(load_status)) as load_status,
initcap(trim(booking_type)) as booking_type
from datawerhouse1.bronze.loads
where load_id is not null;
describe table datawerhouse1.silver.loads;
select * from datawerhouse1.silver.loads

In [0]:
create or replace table datawerhouse1.silver.maintenance_records as
SELECT
    trim(maintenance_id) as maintenance_id,
    trim(truck_id) as truck_id,
    maintenance_date,
    upper(trim(maintenance_type)) as maintenance_type,
    odometer_reading,
    round(labor_hours,2) as labor_hours,
    round(labor_cost,2) as labor_cost,
    round(parts_cost,2) as parts_cost,
    round(total_cost,2) as total_cost,
     downtime_hours,
    upper(trim(facility_location)) as facility_location,
    initcap(trim(service_description)) as service_description
FROM datawerhouse1.bronze.maintenance_records
WHERE maintenance_id is not null
  AND truck_id is not null;
  describe table datawerhouse1.silver.maintenance_records;
select * from datawerhouse1.silver.maintenance_records

In [0]:
create or replace table datawerhouse1.silver.routes as
SELECT
    trim(route_id) as route_id,
    upper(trim(origin_city)) as origin_city,
    origin_state,
    upper(trim(destination_city)) as destination_city,
    destination_state,
    typical_distance_miles,
    round(cast(trim(base_rate_per_mile) as double), 2) as base_rate_per_mile,
    round(cast(trim(fuel_surcharge_rate) as double), 2) as fuel_surcharge_rate,
    typical_transit_days
FROM datawerhouse1.bronze.routes
WHERE route_id is not null;
select * from datawerhouse1.silver.routes

In [0]:
create or replace table datawerhouse1.silver.safety_incidents as
SELECT
    trim(incident_id) as safety_incident_id,
    trim(truck_id) as truck_id,
    incident_date,
    incident_type,
    upper(trim(location_city)) as location_city,
    location_state,
    initcap(trim(at_fault_flag)) as at_fault_flag,
    initcap(trim(injury_flag)) as injury_flag,
    round(cast(trim(vehicle_damage_cost) as double), 1) as vehicle_damage_cost,
    round(cast(trim(cargo_damage_cost) as double), 1) as cargo_damage_cost,
    round(cast(trim(claim_amount) as double), 1) as claim_amount,
    initcap(trim(preventable_flag)) as  preventable_flag,
    initcap(trim(description)) as description
FROM datawerhouse1.bronze.safety_incidents
WHERE incident_id is not null
  AND truck_id is not null;
  select * from datawerhouse1.silver.safety_incidents

In [0]:
create or replace table datawerhouse1.silver.trailers as
select
 trim(trailer_id) as trailer_id,
 trailer_number,
 initcap(trim(trailer_type)) as trailer_type,
 length_feet,
 model_year,
 vin,
 acquisition_date,
 initcap(trim(status)) as status,
 upper(trim(current_location)) as current_location
 from datawerhouse1.bronze.trailers
where trailer_id is not null;
select * from datawerhouse1.silver.trailers

In [0]:
create or replace table datawerhouse1.silver.trips as
SELECT
    trim(trip_id) as trip_id,
    trim(load_id) as load_id,
    -- Handle null values in driver_id
    coalesce(trim(driver_id), 'UNKNOWN') as driver_id,
    -- Handle null values in truck_id
    coalesce(trim(truck_id), 'UNKNOWN') as truck_id,
    --- Handle null values in trailer_id
    coalesce(trim(trailer_id), 'UNKNOWN') as trailer_id,
    dispatch_date,
    round(cast(trim(actual_distance_miles) as double), 1) as actual_distance_miles,
    round(cast(trim(actual_duration_hours) as double), 1) as actual_duration_hours,
    round(cast(trim(fuel_gallons_used) as double), 1) as fuel_gallons_used,
    round(cast(trim(average_mpg) as double), 1) as average_mpg,
    round(cast(trim(idle_time_hours) as double), 1) as idle_time_hours,
    initcap(trim(trip_status)) as trip_status
FROM datawerhouse1.bronze.trips
WHERE trip_id is not null;

select driver_id,truck_id,trailer_id from datawerhouse1.silver.trips
where driver_id = 'UNKNOWN' or truck_id = 'UNKNOWN' or trailer_id = 'UNKNOWN'
group by driver_id,truck_id,trailer_id

In [0]:
create or replace table datawerhouse1.silver.truck_utilization_metrics as
select
       trim(truck_id) as truck_id,
       month,
       trips_completed,
       total_miles,
       round(total_revenue, 2) as total_revenue,
       round(average_mpg, 2) as average_mpg,
       maintenance_events,
       round(maintenance_cost, 2) as maintenance_cost,
       round(downtime_hours, 2) as downtime_hours,
       round(utilization_rate, 4) as utilization_rate
from datawerhouse1.bronze.truck_utilization_metrics
where truck_id is not null;
select * from datawerhouse1.silver.truck_utilization_metrics

       

In [0]:
create or replace table datawerhouse1.silver.trucks as
select
 trim(truck_id) as truck_id,
unit_number,
initcap(trim(make)) as  make,
model_year,
vin,
acquisition_date,
round(cast(trim(acquisition_mileage) as double),2) as acquisition_mileage,
initcap(trim(fuel_type)) as fuel_type,
round(cast(trim(tank_capacity_gallons) as double),2) as tank_capacity_gallons,
initcap(trim(status)) as status,
upper(trim(home_terminal)) as home_terminal
from datawerhouse1.bronze.trucks
where truck_id is not null;
select * from datawerhouse1.silver.trucks

In [0]:
-- Get only one record per driver_id (primary key: driver_id, month)
select * from (
  select *,
    row_number() over(partition by driver_id, month order by month desc) as r
  from datawerhouse1.silver.driver_monthly_metrics
) j
where r = 1;

-- Get latest record per driver_id
select * from(  
  select *,      
    row_number() over(partition by driver_id order by month desc) as r
  from datawerhouse1.silver.driver_monthly_metrics
) j
where r = 1;
--check for unwanted spaces -----
select * from datawerhouse1.silver.driver_monthly_metrics
where driver_id != trim(driver_id)
